In [ ]:
import sys
sys.path.append("../")  # Ensure parent directory is in sys.path
from utils import *
from utils.custom_classes import ParameterField
from utils.sigmav_functions import *
import numpy as np
import matplotlib.pyplot as plt
import itertools
import plotly.graph_objects as go
from tqdm import tqdm

# Parametrization points

In [ ]:
# import parameter set from config file
from inputs.config import *

# Perform the parametric analysis

In [3]:
input_data = [
    V_plasma_field.data,
    T_i_field.data,
    n_tot_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
    
    I_target_field.data,
    
    TBR_DT_field.data,
    TBR_DDn_field.data,
    
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]


In [4]:

# Create iterator based only on the number of parameter variations (first dimension)
param_ranges = [range(data.shape[0]) for data in input_data]

results = []

for param_combo in tqdm(itertools.product(*param_ranges), 
                       total=np.prod([data.shape[0] for data in input_data]), 
                       desc="Parametric analysis"):
    
    # Extract data for each parameter
    extracted_data = [input_data[i][param_idx] for i, param_idx in enumerate(param_combo)]
        
    # Unpack the extracted data
    (V_plasma, T_i, n_tot,
     tau_p_T, tau_p_He3, 
     P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT,
     
     I_target,
     
     TBR_DT, TBR_DDn,
     
     eta_th, plant_avail, Cost_per_kWh,
     ) = extracted_data

    # Get cross-sections
    sigmav_DD = sigmav_DD_BoschHale(T_i)[0].to('m^3/s')  # [m^3/s]
    sigmav_DD_p = sigmav_DD_BoschHale(T_i)[1].to('m^3/s')  # [m^3/s]
    sigmav_DD_n = sigmav_DD_BoschHale(T_i)[2].to('m^3/s')  # [m^3/s]
    sigmav_DT = sigmav_DT_BoschHale(T_i).to('m^3/s')    # [m^3/s]
    sigmav_DHe3 = sigmav_DHe3_BoschHale(T_i).to('m^3/s')   # [m^3/s]
    
    # Calculate the reaction rates
    DD_reaction_rates = calculate_reaction_rates_DD(n_tot, T_i, V_plasma, tau_p_T, tau_p_He3)
    # ESTIMATE TRITIUM PRODUCTION
    Tdot_fusion, Tdot_breedingDT, Tdot_breedingDD, Tdot_tot = compute_tritium_production(DD_reaction_rates, TBR_DT, TBR_DDn, V_plasma, tau_p_T)
    
    # CALCULATE THE STARTUP TIME
    t_startup = compute_startup_time(I_target, Tdot_tot, molecular_weight_T)
    
    # CALCULATE THE FUSION POWER
    P_DD, P_DT, P_DHe3, P_DD_tot, P_DT_full = compute_fusion_power(DD_reaction_rates, n_tot, T_i, V_plasma)
            
    # CALCULATE THE NET ELECTRICAL POWER and $ lost (comparing DD and 50D50T operation)
    P_e_net_DD, Q_DD = calculate_P_e_net((P_DD+P_DT+P_DHe3),P_aux=P_aux, P_rad = P_lost_rad,  plant_avail=plant_avail, eta_th=eta_th)
    P_e_net_DT_full, Q_DT_full = calculate_P_e_net(P_DT_full,P_aux=P_aux_all_DT, P_rad = P_lost_rad_all_DT,  plant_avail=plant_avail, eta_th=eta_th)
    E_e_net_DD = P_e_net_DD*t_startup
    E_e_net_DT_full = P_e_net_DT_full*t_startup
    E_lost = E_e_net_DT_full - E_e_net_DD
    Dollar_Lost = E_lost * Cost_per_kWh

    T_i_profile_avg = np.mean(T_i).to('keV').magnitude
    n_tot_profile_avg = np.mean(n_tot).to('1/meter**3').magnitude
    
    row = [
        # INPUTS
        V_plasma.to('m^3'),                     # 0
        tau_p_T.to('s'),                        # 1
        tau_p_He3.to('s'),                      # 2
        P_aux.to('MW'),                         # 3
        P_aux_all_DT.to('MW'),                  # 4
        P_lost_rad.to('MW'),                    # 5
        P_lost_rad_all_DT.to('MW'),             # 6
        T_i.to('keV'),                          # 7
        n_tot.to('m^-3'),                       # 8
        #injection_rate_max,                    # -
        TBR_DT,                                 # 9
        TBR_DDn,                                # 10
        I_target,                               # 11
        eta_th,                                 # 12
        plant_avail,                            # 13
        Cost_per_kWh.to('1/kWh'),               # 14
        # OUTPUTS
        sigmav_DT.to('m^3/s'),                  # 16
        sigmav_DD_n.to('m^3/s'),                # 17
        sigmav_DD_p.to('m^3/s'),                # 18
        sigmav_DHe3.to('m^3/s'),                # 19
        P_DT.to('MW'),                          # 20
        P_DD.to('MW'),                         # 21
        P_DHe3.to('MW'),                         # 22
        P_DT_full.to('MW'),                     # 23
        P_e_net_DD.to('MW'),                    # 24
        Q_DD,                                   # 25
        P_e_net_DT_full.to('MW'),               # 26
        Q_DT_full,                              # 27
        E_e_net_DD.to('MJ'),                    # 28
        E_e_net_DT_full.to('MJ'),               # 29
        t_startup.to('hour'),                   # 30
        E_lost.to('MJ'),                        # 31
        Dollar_Lost.to(''),                     # 32
    ]
    results.append(row)

Parametric analysis: 100%|██████████| 759375/759375 [3:12:10<00:00, 65.86it/s]   


In [ ]:
#save results to a CSV file
import pandas as pd
columns = [
    # INPUTS
    "V_plasma (m^3)",                     
    "tau_p_T (s)",                        
    "tau_p_He3 (s)",                      
    "P_aux (MW)",                         
    "P_aux_all_DT (MW)",                  
    "P_lost_rad (MW)",                    
    "P_lost_rad_all_DT (MW)",             
    "T_i (keV)",                          
    "n_tot (m^-3)",                       
    #"injection_rate_max",                    
    "TBR_DT",                                 
    "TBR_DDn",                                
    "I_target",                               
    "eta_th",                                 
    "plant_avail",                            
    "Cost_per_kWh (1/kWh)",               
    # OUTPUTS
    "sigmav_DT (m^3/s)",                  
    "sigmav_DD_n (m^3/s)",                
    "sigmav_DD_p (m^3/s)",                
    "sigmav_DHe3 (m^3/s)",                
    "P_DT (MW)",                          
    "P_DD (MW)",                         
    "P_DHe3 (MW)",                         
    "P_DT_full (MW)",                     
    "P_e_net_DD (MW)",                    
    "Q_DD",                                   
    "P_e_net_DT_full (MW)",               
    "Q_DT_full",                              
    "E_e_net_DD (MJ)",                    
    "E_e_net_DT_full (MJ)",               
    "t_startup (hour)",                   
    "E_lost (MJ)",                        
    "Dollar_Lost ($)",                     
]
df = pd.DataFrame(results, columns=columns)
output_filename = "parametric_analysis_I_target_results.csv"
df.to_csv(output_filename, index=False)

In [1]:
print("Parametric analysis completed.")
# print dollar lost for each case
for i, row in enumerate(results):
    print(f"Case {i+1}: time {row[30]/24:.2f} days {row[-1]:.2f} $")

Parametric analysis completed.


NameError: name 'results' is not defined